In [1]:
import torch;
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
%%writefile dataset.py
import os
import numpy as np
import torch
from torch.utils.data import Dataset
import random

class NpySpectrogramDataset(Dataset):
    def __init__(self, root, mel_bins=80, expected_width=None, class_to_label=None,
        aug_ratio=0.0, use_noise=False, use_chop=False, use_fast=False, use_slow=False,
        seed=42):

        self.root = root
        self.mel_bins = mel_bins
        self.expected_width = expected_width

        if class_to_label is None:
            self.class_to_label = {"class_0": 0, "class_1": 1}
        else:
            self.class_to_label = class_to_label

        rng = random.Random(seed)

        original_samples = []
        aug_samples = []

        for class_name, label in self.class_to_label.items():
            class_dir = os.path.join(root, class_name)
            for speaker in os.listdir(class_dir):
                speaker_dir = os.path.join(class_dir, speaker)
                for filename in os.listdir(speaker_dir):
                    if filename.endswith(".npy"):
                        full_path = os.path.join(speaker_dir, filename)
                        name = filename.lower()
                        is_noise = "_noise_" in name
                        is_chop  = "_chop_"  in name
                        is_fast  = "_fast_"  in name
                        is_slow  = "_slow_"  in name
                        is_aug = is_noise or is_chop or is_fast or is_slow
                        if is_aug:
                            if (is_noise and use_noise) or (is_chop and use_chop) or (is_fast and use_fast) or (is_slow and use_slow):
                                aug_samples.append((full_path, label))
                        else:
                            original_samples.append((full_path, label))

        if 0.0 <= aug_ratio < 1.0 and len(aug_samples) > 0:
            k = int(round(len(aug_samples) * aug_ratio))
            aug_samples = rng.sample(aug_samples, k)

        self.samples = original_samples + aug_samples
        self.samples.sort()

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        path, label = self.samples[index]
        arr = np.load(path)

        if arr.ndim != 2:
            arr = np.squeeze(arr)

        H, W = arr.shape
        if H != self.mel_bins:
            raise ValueError(f"{path} mel bins {H}, expected {self.mel_bins}")
        if W != self.expected_width:
            raise ValueError(f"{path} width {W}, expected {self.expected_width}")
        
        x = torch.from_numpy(arr.astype(np.float32)).unsqueeze(0)  # [1, 80, W]
        y = torch.tensor(label, dtype=torch.long)
        return x, y

Writing dataset.py


In [3]:
%%writefile model.py
import torch
import torch.nn as nn
from torchvision import models


class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes,kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(planes, planes,kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.downsample = downsample

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out = out + identity
        out = self.relu(out)

        return out


class ResNet18Mel(nn.Module):
    def __init__(self, pretrained):
        super().__init__()

        self.inplanes = 64

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(planes=64,  blocks=2, stride=1)
        self.layer2 = self._make_layer(planes=128, blocks=2, stride=2)
        self.layer3 = self._make_layer(planes=256, blocks=2, stride=2)
        self.layer4 = self._make_layer(planes=512, blocks=2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, 2)

        if pretrained:
            self._load_pretrained_resnet18_weights()

    def _make_layer(self, planes, blocks, stride = 1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
        layers = []
        layers.append(BasicBlock(in_planes=self.inplanes, planes=planes,stride=stride, downsample=downsample))
        self.inplanes = planes

        for _ in range(1, blocks):
            layers.append(BasicBlock(in_planes=self.inplanes,planes=planes,stride=1,downsample=None))

        return nn.Sequential(*layers)

    def _load_pretrained_resnet18_weights(self):
        try:
            tv_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        except AttributeError: 
            tv_model = models.resnet18(pretrained=True)

        tv_state = tv_model.state_dict()
        for k in ["conv1.weight", "fc.weight", "fc.bias"]:
            if k in tv_state:
                tv_state.pop(k)
        self.load_state_dict(tv_state, strict=False)

        with torch.no_grad():
            conv1_rgb = tv_model.conv1.weight        
            conv1_gray = conv1_rgb.mean(dim=1, keepdim=True)  
            self.conv1.weight.copy_(conv1_gray)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)         
        x = torch.flatten(x, 1)    
        x = self.fc(x)               
        return x


def resnet18(pretrained):
    return ResNet18Mel(pretrained=pretrained)

Writing model.py


In [4]:
%%writefile train.py
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
from math import inf
import random
import numpy as np
from model import resnet18
from dataset import NpySpectrogramDataset
from tensorboardX import SummaryWriter

IMG_H = 80
IMG_W = 300
BATCH_SIZE = 32
EPOCHS = 20

AUG_RATIO=1.0
USE_NOISE=False
USE_CHOP=True
USE_FAST=False
USE_SLOW=False

TRAIN_PATH = "/kaggle/input/spectrograms-aug/train_data"
VALID_PATH = "/kaggle/input/spectrograms-aug/validation_data"
TEST_PATH = "/kaggle/input/spectrograms-aug/test_data"

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    
    
def train_model(model, train_loader, device, criterion, optimizer):
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    model.train()
    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device).long() #converts to int64
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        values, predictions = outputs.max(1)
        train_correct += (predictions == labels).sum().item() #.item() converts tensor sum to python int
        train_total += labels.size(0) 

    train_loss = train_loss / train_total
    train_acc = train_correct / train_total
    print(f"Train Loss: {train_loss:.4f},  Train Acc: {train_acc:.4f}")
    return train_loss, train_acc


def validate_model(model, valid_loader, device, criterion):
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad(): #Validation should NOT compute gradients, update weights
        for inputs, labels in valid_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            values, predictions = outputs.max(1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

        val_loss = val_loss / val_total
        val_acc = val_correct / val_total
        print(f"Validation Loss: {val_loss:.4f},  Validation Acc: {val_acc:.4f}")
        return val_loss, val_acc


def test_model(model, test_loader, device, criterion):
    model.eval()
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            values, predictions = outputs.max(1)
            test_correct += (predictions == labels).sum().item()
            test_total += labels.size(0)

        test_loss /= test_total
        test_acc = test_correct / test_total
        print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
        return test_loss, test_acc

def compute_f1(model, data_loader, device):
    model.eval()
    all_predictions = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).long()

            outputs = model(inputs)
            values, predictions = outputs.max(1)

            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_predictions, average="macro")
    weighted_f1 = f1_score(all_labels, all_predictions, average="weighted")
    return macro_f1, weighted_f1


def main():    
    SEED = 42
    seed_everything(SEED) 

    model_name = "simple_cnn" 
    #Initialize state saving
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "test_loss": [],
        "test_acc": [],
        "val_f1_macro": [],
        "val_f1_weighted": [],
        "test_f1_macro": [],
        "test_f1_weighted": [],
    }

    tensorboard_dir = os.path.join("tensor_board_states", model_name)
    writer = SummaryWriter(log_dir=tensorboard_dir)
    best_val_loss = inf

    checkpoint_dir = os.path.join("checkpoints", model_name)
    os.makedirs(checkpoint_dir, exist_ok=True)
    best_model_path = os.path.join(checkpoint_dir, "best_model.pth")

    #Data Loading 
    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    train_dataset = NpySpectrogramDataset(TRAIN_PATH, mel_bins=IMG_H, expected_width=IMG_W, aug_ratio=AUG_RATIO, use_noise=USE_NOISE, use_chop=USE_CHOP,
    use_fast=USE_FAST,use_slow=USE_SLOW, seed=SEED)
    valid_dataset = NpySpectrogramDataset(VALID_PATH, mel_bins=IMG_H, expected_width=train_dataset.expected_width ,class_to_label=train_dataset.class_to_label)
    test_dataset  = NpySpectrogramDataset(TEST_PATH, mel_bins=IMG_H, expected_width=train_dataset.expected_width ,class_to_label=train_dataset.class_to_label)

    g = torch.Generator()
    g.manual_seed(SEED)

    train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True, num_workers = 2, pin_memory = True, worker_init_fn=seed_worker,generator=g,persistent_workers=True)
    valid_loader = DataLoader(valid_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)
    test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory = True)
    print("Train len:", len(train_dataset))
    print("Valid len:", len(valid_dataset))
    print("Test len:", len(test_dataset))
    model = resnet18(pretrained=False).to(device)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-3) #model.parameters() = all weights
    
    #Train
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_model(model, train_loader, device, criterion, optimizer)

        history["train_loss"].append(train_loss)  
        history["train_acc"].append(train_acc)    
        writer.add_scalar("train_loss", train_loss, epoch+1)  
        writer.add_scalar("train_acc",  train_acc,  epoch+1) 

        #Validate
        val_loss, val_acc = validate_model(model, valid_loader, device, criterion)

        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        writer.add_scalar("val_loss", val_loss, epoch+1)
        writer.add_scalar("val_acc",  val_acc,  epoch+1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save({
                "epoch": epoch,                                 
                "model_state": model.state_dict(),              
                "optimizer_state": optimizer.state_dict(),     
                "val_loss": val_loss, 
                "val_acc": val_acc,
                "img_shape": (IMG_H, IMG_W),                    
                "class_to_label": train_dataset.class_to_label,     
            }, best_model_path)

    #Test
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state"])

    test_loss, test_acc = test_model(model, test_loader, device, criterion)

    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)
    writer.add_scalar("test_loss", test_loss, EPOCHS)
    writer.add_scalar("test_acc",  test_acc,  EPOCHS)

    val_macro_f1, val_weighted_f1 = compute_f1(model, valid_loader, device)
    test_macro_f1, test_weighted_f1 = compute_f1(model, test_loader, device)

    print(f"Valid. macro averaged f1-score: {val_macro_f1}")
    print(f"Valid. weighted f1-score: {val_weighted_f1}")
    print(f"Test macro averaged f1-score: {test_macro_f1}")
    print(f"Test weighted f1-score: {test_weighted_f1}")
    
    history["val_f1_macro"].append(val_macro_f1)
    history["val_f1_weighted"].append(val_weighted_f1)
    history["test_f1_macro"].append(test_macro_f1)
    history["test_f1_weighted"].append(test_weighted_f1)

    history_path = os.path.join(checkpoint_dir, "history.pt")
    torch.save(history, history_path)  
    writer.close()


if __name__ == "__main__":
    main()

Writing train.py


In [5]:
!pip install tensorboardX

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.1 MB/s eta 0:00:00


In [6]:
!python train.py

Train len: 7654
Valid len: 543
Test len: 688
Train Loss: 0.4747,  Train Acc: 0.8124
Validation Loss: 0.9522,  Validation Acc: 0.5893
Train Loss: 0.3332,  Train Acc: 0.9194
Validation Loss: 0.6602,  Validation Acc: 0.7127
Train Loss: 0.2945,  Train Acc: 0.9477
Validation Loss: 0.6175,  Validation Acc: 0.7201
Train Loss: 0.2631,  Train Acc: 0.9683
Validation Loss: 0.7673,  Validation Acc: 0.6722
Train Loss: 0.2565,  Train Acc: 0.9710
Validation Loss: 0.6641,  Validation Acc: 0.6924
Train Loss: 0.2399,  Train Acc: 0.9828
Validation Loss: 0.6753,  Validation Acc: 0.6961
Train Loss: 0.2526,  Train Acc: 0.9739
Validation Loss: 0.5731,  Validation Acc: 0.7624
Train Loss: 0.2323,  Train Acc: 0.9845
Validation Loss: 0.5965,  Validation Acc: 0.7753
Train Loss: 0.2392,  Train Acc: 0.9797
Validation Loss: 0.7508,  Validation Acc: 0.6483
Train Loss: 0.2325,  Train Acc: 0.9842
Validation Loss: 0.7151,  Validation Acc: 0.6943
Train Loss: 0.2324,  Train Acc: 0.9835
Validation Loss: 0.6120,  Validation